# InfoGuard — Phase 4 Results
## Reproducing Table II from Bayiz & Topcu (2022)

This notebook loads the output of `pipeline/run_pipeline.py` and reproduces Table II
from the paper, plus additional diagnostics.

**Run the pipeline first:**
```bash
# Quick validation (synthetic, no data needed)
python -m pipeline.run_pipeline --synthetic

# Full WICO evaluation (500 samples)
python -m pipeline.run_pipeline --n-samples 500 --label-source wico_folders
```

**Paper Table II (target numbers):**

| α | λ | E[R∞] true | E[R∞] false | P[R∞<5] true | P[R∞<5] false |
|---|---|---|---|---|---|
| — | — | 50.1 | 48.7 | 0.01 | 0.00 | ← control |
| 1.5 | 1 | 32.8 | 26.1 | 0.05 | 0.13 |
| 2 | 1.5 | 39.4 | 28.2 | 0.04 | 0.09 |
| 3 | 2 | 41.1 | 36.0 | 0.00 | 0.02 |


In [ ]:
import sys, os
from pathlib import Path

# Navigate to project root from evaluation/
ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

from config import cfg

EVAL_DIR = Path(cfg.paths.evaluation)
FIG_DIR  = EVAL_DIR / 'pipeline_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Paper Table II reference values
PAPER_TABLE2 = {
    'control':      {'E_true': 50.1, 'E_false': 48.7, 'P_true': 0.01, 'P_false': 0.00},
    'α=1.5,λ=1.0':  {'E_true': 32.8, 'E_false': 26.1, 'P_true': 0.05, 'P_false': 0.13},
    'α=2.0,λ=1.5':  {'E_true': 39.4, 'E_false': 28.2, 'P_true': 0.04, 'P_false': 0.09},
    'α=3.0,λ=2.0':  {'E_true': 41.1, 'E_false': 36.0, 'P_true': 0.00, 'P_false': 0.02},
}

print('EVAL_DIR:', EVAL_DIR)
print('Files present:', [f.name for f in EVAL_DIR.glob('pipeline_*.csv')])

## 1. Load Results

In [ ]:
raw_path    = EVAL_DIR / 'pipeline_results_raw.csv'
table2_path = EVAL_DIR / 'pipeline_table2.csv'

if not raw_path.exists():
    raise FileNotFoundError(
        f'Results not found at {raw_path}\n'
        'Run: python -m pipeline.run_pipeline --n-samples 500 --label-source wico_folders'
    )

df = pd.read_csv(raw_path)
t2 = pd.read_csv(table2_path)

print(f'Raw results: {len(df)} rows')
print(f'Unique (α,λ) settings: {df["pair_label"].unique().tolist()}')
print(f'\nSamples per setting: {df.groupby(["pair_label","label"]).size().unstack()}')

## 2. Table II — Reproduction vs Paper

Primary evaluation. Column-by-column comparison with the paper's Table II.

In [ ]:
def styled_table2(t2: pd.DataFrame) -> pd.DataFrame:
    """Display Table II with paper values alongside for comparison."""
    rows = []
    for _, row in t2.iterrows():
        pl = row['pair_label']
        paper = PAPER_TABLE2.get(pl, {})
        rows.append({
            'Setting':         pl,
            'E[R∞] true':      f"{row['E_R_inf_true']:.1f}  (paper: {paper.get('E_true', '—')})",
            'E[R∞] false':     f"{row['E_R_inf_false']:.1f}  (paper: {paper.get('E_false', '—')})",
            'P[R∞<5] true':    f"{row['P_R_inf_lt5_true']:.2f}  (paper: {paper.get('P_true', '—')})",
            'P[R∞<5] false':   f"{row['P_R_inf_lt5_false']:.2f}  (paper: {paper.get('P_false', '—')})",
            'n samples':       int(row['n_samples_true']),
        })
    return pd.DataFrame(rows).set_index('Setting')

comparison = styled_table2(t2)
print('=== Table II: Reproduction vs Paper ===')
display(comparison)

# Key metrics
ctrl = t2[t2['pair_label'] == 'control'].iloc[0]
best = t2[t2['pair_label'] == 'α=1.5,λ=1.0']
if not best.empty:
    best = best.iloc[0]
    false_reduction = (1 - best['E_R_inf_false'] / ctrl['E_R_inf_false']) * 100
    true_reduction  = (1 - best['E_R_inf_true']  / ctrl['E_R_inf_true'])  * 100
    print(f'\nAt α=1.5, λ=1.0:')
    print(f'  False cascade reduction: {false_reduction:.1f}%  (paper target: ~45%)')
    print(f'  True  cascade reduction: {true_reduction:.1f}%  (paper target: ~35%)')
    print(f'  Discrimination: false reduced MORE than true → ✓' if false_reduction > true_reduction else '  ✗ true reduced more than false')

## 3. Figure A — E[R∞] Bar Chart (True vs False, All Settings)

Direct visual equivalent of Table II. The key result is that the orange bars
(false content) are suppressed more than the green bars (true content) for
every (α, λ) setting.

In [ ]:
settings  = t2['pair_label'].tolist()
e_true    = t2['E_R_inf_true'].tolist()
e_false   = t2['E_R_inf_false'].tolist()

x = np.arange(len(settings))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_t = ax.bar(x - width/2, e_true,  width, color='#1D9E75', alpha=0.85,
                label='True content',        edgecolor='white')
bars_f = ax.bar(x + width/2, e_false, width, color='#E24B4A', alpha=0.85,
                label='False content',       edgecolor='white')

# Paper reference lines
ax.axhline(PAPER_TABLE2['control']['E_true'],  color='#1D9E75', linestyle='--',
           linewidth=1.0, alpha=0.6, label='Paper control (true)')
ax.axhline(PAPER_TABLE2['control']['E_false'], color='#E24B4A', linestyle='--',
           linewidth=1.0, alpha=0.6, label='Paper control (false)')

# Value labels
for bar in bars_t:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
for bar in bars_f:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(settings, rotation=10, fontsize=9)
ax.set_ylabel('E[R∞] — Expected cascade size', fontsize=11)
ax.set_title(
    'Table II Reproduction: Expected Cascade Size by LP Setting\n'
    'Algorithm 2 suppresses false content more than true content',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
out = FIG_DIR / 'table2_cascade_size_bars.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', out)

## 4. Figure B — Cascade Size Distributions

Histograms of R∞ across all samples for each (α, λ) setting.
The LP shifts the false distribution leftward while keeping the true
distribution near the control.

In [ ]:
settings_to_plot = [s for s in settings if s != 'control']
n_plots = len(settings_to_plot)

fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4), sharey=True)
if n_plots == 1:
    axes = [axes]

ctrl_false = df[(df['pair_label']=='control') & (df['label']=='false')]['cascade_size']
ctrl_true  = df[(df['pair_label']=='control') & (df['label']=='true') ]['cascade_size']

for ax, setting in zip(axes, settings_to_plot):
    alg_false = df[(df['pair_label']==setting) & (df['label']=='false')]['cascade_size']
    alg_true  = df[(df['pair_label']==setting) & (df['label']=='true') ]['cascade_size']

    bins = np.linspace(0, max(ctrl_false.max(), ctrl_true.max(), 5) + 1, 25)

    ax.hist(ctrl_false,  bins=bins, alpha=0.3, color='#E24B4A', label='False (control)')
    ax.hist(ctrl_true,   bins=bins, alpha=0.3, color='#1D9E75', label='True  (control)')
    ax.hist(alg_false,   bins=bins, alpha=0.7, color='#E24B4A', label='False (alg2)')
    ax.hist(alg_true,    bins=bins, alpha=0.7, color='#1D9E75', label='True  (alg2)')

    ax.axvline(5, color='#333333', linestyle=':', linewidth=1.2, label='R∞=5 threshold')
    ax.set_title(f'{setting}\nE[false]={alg_false.mean():.1f} | E[true]={alg_true.mean():.1f}',
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('Cascade size R∞', fontsize=9)
    ax.spines[['top','right']].set_visible(False)

axes[0].set_ylabel('Count', fontsize=9)
axes[0].legend(fontsize=7)

plt.suptitle(
    'Cascade Size Distributions: Control vs Algorithm 2\n'
    'Shaded = control; solid = with dropout. False distribution shifts left.',
    fontsize=11, fontweight='bold', y=1.03
)
plt.tight_layout()
out = FIG_DIR / 'cascade_size_distributions.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', out)

## 5. Figure C — LP Feasibility and Discrimination Gap

Two metrics the paper discusses:
1. **Discrimination** = E[R∞_false] / E[R∞_true] — how much more false is suppressed
   than true. Lower = better discrimination. Control = ~1.0 (equal).
2. **True content preservation ratio** = E[R∞_true under algo] / E[R∞_true control]
   Should stay high (close to 1.0) to confirm the LP constraint is effective.

In [ ]:
ctrl_e_true  = float(t2[t2['pair_label']=='control']['E_R_inf_true'].iloc[0])
ctrl_e_false = float(t2[t2['pair_label']=='control']['E_R_inf_false'].iloc[0])

metric_rows = []
for _, row in t2.iterrows():
    pl = row['pair_label']
    et = row['E_R_inf_true']
    ef = row['E_R_inf_false']
    metric_rows.append({
        'Setting':              pl,
        'Discrimination\n(false/true ratio)': ef / max(et, 0.1),
        'True preservation\n(algo/control)':  et / max(ctrl_e_true, 0.1),
        'False suppression\n(1 - algo/ctrl)': 1 - ef / max(ctrl_e_false, 0.1),
    })
metrics_df = pd.DataFrame(metric_rows)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#888780'] + ['#7F77DD'] * (len(metric_rows) - 1)

for ax, col in zip(axes, [
    'Discrimination\n(false/true ratio)',
    'True preservation\n(algo/control)',
    'False suppression\n(1 - algo/ctrl)',
]):
    vals = metrics_df[col].tolist()
    lbls = metrics_df['Setting'].tolist()
    bars = ax.bar(range(len(vals)), vals, color=colors, edgecolor='white')
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{v:.2f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(range(len(lbls)))
    ax.set_xticklabels(lbls, rotation=12, fontsize=8)
    ax.set_title(col.replace('\n', ' '), fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

# Reference lines
axes[0].axhline(1.0, color='#E24B4A', linestyle='--', linewidth=1, label='No discrimination')
axes[0].legend(fontsize=8)
axes[1].axhline(1.0, color='#888780', linestyle='--', linewidth=1, label='No change')
axes[1].legend(fontsize=8)

plt.suptitle(
    'Algorithm 2 Performance Metrics\n'
    'Discrimination < 1 = false more suppressed than true (paper goal)',
    fontsize=11, fontweight='bold', y=1.02
)
plt.tight_layout()
out = FIG_DIR / 'algorithm2_metrics.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', out)

## 6. Figure D — Alpha Trade-off

How increasing α (the true-content safety parameter) affects both
false suppression and true preservation. Reproduces the qualitative
pattern from paper Section V.A: higher α preserves true content more
but also allows more false content to survive.

In [ ]:
algo_rows = t2[t2['pair_label'] != 'control'].copy()
algo_rows['alpha_float'] = pd.to_numeric(algo_rows['alpha'], errors='coerce')
algo_rows = algo_rows.sort_values('alpha_float')

if len(algo_rows) >= 2:
    fig, ax = plt.subplots(figsize=(7, 4))

    alphas = algo_rows['alpha_float'].tolist()
    ax.plot(alphas, algo_rows['E_R_inf_true'].tolist(),
            'o-', color='#1D9E75', linewidth=2, markersize=8, label='True content E[R∞]')
    ax.plot(alphas, algo_rows['E_R_inf_false'].tolist(),
            's-', color='#E24B4A', linewidth=2, markersize=8, label='False content E[R∞]')

    # Paper reference
    ax.axhline(PAPER_TABLE2['control']['E_true'],  color='#1D9E75',
               linestyle=':', linewidth=1.2, alpha=0.5, label='Control true')
    ax.axhline(PAPER_TABLE2['control']['E_false'], color='#E24B4A',
               linestyle=':', linewidth=1.2, alpha=0.5, label='Control false')

    ax.set_xlabel('α (LP safety parameter)', fontsize=11)
    ax.set_ylabel('E[R∞]', fontsize=11)
    ax.set_title(
        'α Trade-off: Higher α preserves true content\nbut allows more false content to survive',
        fontsize=11, fontweight='bold'
    )
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    out = FIG_DIR / 'alpha_tradeoff.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved:', out)
else:
    print('Need at least 2 alpha settings to plot trade-off.')

## 7. Synthetic Table I Validation (if available)

Loads results from `--synthetic` run, which reproduces Table I
(synthetic SBM experiments). This validates the LP optimizer in isolation
without any real data.

In [ ]:
synth_path = EVAL_DIR / 'synthetic_table1_k2.csv'
if synth_path.exists():
    syn = pd.read_csv(synth_path)
    print('Synthetic Table I summary (k=2):')

    summary = syn.groupby(['alpha','lambda']).agg(
        mean_E_false  = ('E_R_inf_N_false', 'mean'),
        mean_E_true   = ('E_R_inf_N_true',  'mean'),
        mean_P_false  = ('P_lt_N10_false',  'mean'),
        mean_P_true   = ('P_lt_N10_true',   'mean'),
    ).reset_index().round(3)

    # Paper Table I (balanced 2-partition α=1.5 λ=1):
    # E[R∞/N] false=0.32, true=0.51, P[R∞<N/10] false=0.43, true=0.28
    display(summary)

    PAPER_T1 = [
        {'alpha': 1.5, 'lambda': 1.0, 'E_false': 0.32, 'E_true': 0.51,
         'P_false': 0.43, 'P_true': 0.28},
        {'alpha': 2.0, 'lambda': 1.5, 'E_false': 0.41, 'E_true': 0.72,
         'P_false': 0.30, 'P_true': 0.18},
    ]
    print('\nPaper Table I (balanced 2-partition):')
    display(pd.DataFrame(PAPER_T1))
else:
    print(f'Synthetic results not found at {synth_path}')
    print('Run: python -m pipeline.run_pipeline --synthetic --synthetic-k 2')

## 8. Summary

Expected conclusions (will be confirmed by your numbers):

1. **False content is suppressed more than true** across all (α, λ) settings —
   the core claim of the paper. The LP exploits the b⁺ vs b⁻ asymmetry.

2. **Higher α preserves true content better** but allows more false to survive —
   the trade-off described in Section V of the paper.

3. **P[R∞ < 5] for false content increases** with the algorithm — many false
   cascades are killed before they reach 5 users, matching Table II.

4. **WICO cascades are smaller** than the paper's synthetic examples (~50 nodes
   vs 1000) because WICO graphs are capped at 100 nodes by dataset construction.
   The qualitative pattern (false > true suppression) still holds.


In [ ]:
print('=== Final Summary ===')
print(f'Total simulations : {len(df)}')
print(f'Samples per setting: {len(df) // (len(df["pair_label"].unique()) * 2)}')
print()

# Final Table II comparison
print('Reproduced Table II:')
display(t2[['pair_label','E_R_inf_true','E_R_inf_false',
            'P_R_inf_lt5_true','P_R_inf_lt5_false']].rename(columns={
    'pair_label':'Setting',
    'E_R_inf_true':'E[R∞] true',
    'E_R_inf_false':'E[R∞] false',
    'P_R_inf_lt5_true':'P[R∞<5] true',
    'P_R_inf_lt5_false':'P[R∞<5] false',
}))

print()
print('Paper Table II:')
display(pd.DataFrame([
    {'Setting': k, 'E[R∞] true': v['E_true'], 'E[R∞] false': v['E_false'],
     'P[R∞<5] true': v['P_true'], 'P[R∞<5] false': v['P_false']}
    for k, v in PAPER_TABLE2.items()
]))

print()
print('Figures saved to:', FIG_DIR)